In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch
import os
import json

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "artifacts").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate the MIRx project root from {start}")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
BATCH_SIZE = 128

In [ ]:
path = DATA_DIR / "processed" / "preprocessed_medicine_dataset.csv"
df = pd.read_csv(path)
df.head()

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
df.info()

In [ ]:
print(df.columns.tolist())

In [ ]:
TEXT_COLUMNS = ['name', 'Chemical Class', 'Habit Forming', 'Therapeutic Class', 'Action Class', 
                'Substitutes', 'Side Effects', 'Uses']

In [ ]:
def create_document(row):
    parts = {}

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.notna(value):
            value = str(value).strip()

            if value:
                parts[column] = value

    return " ".join([f"{k}: {v}" for k, v in parts.items()])

df["document"] = df.apply(create_document, axis=1)

In [ ]:
df.head()

In [ ]:
df["document"][0]

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
print("Model embedding dimension: ", model.get_embedding_dimension())

In [ ]:
print(model.device)

In [ ]:
documents = df["document"].tolist()

embeddings = model.encode(
    documents,
    show_progress_bar = True,
    batch_size = BATCH_SIZE,
    convert_to_numpy = True,
    normalize_embeddings = True 
)

print("Embeddings shape:", embeddings.shape)

In [ ]:
print("Shape:", embeddings.shape)
print("dtype:", embeddings.dtype)

print("\nFirst embedding:")
print(embeddings[0][:20])

In [ ]:
norms = np.linalg.norm(embeddings[:100], axis=1)

print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())

In [ ]:
embedding_path = os.path.join(ARTIFACTS_DIR, "embeddings.npy")

np.save(embedding_path, embeddings)

print(f"Saved embeddings to: {embedding_path}")

In [ ]:
metadata = df.drop(columns=["document"]).copy()
metadata["document_id"] = "doc_" + pd.Series(np.arange(len(metadata))).astype(str).str.zfill(6)

metadata["document"] = df["document"]

metadata_path = os.path.join(ARTIFACTS_DIR, "metadata.parquet")

metadata.to_parquet(metadata_path, index=False)

print(f"Saved metadata to: {metadata_path}")

In [ ]:
config = {
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": int(model.get_embedding_dimension()),
    "normalized": True,
    "batch_size": BATCH_SIZE,
    "num_documents": len(df),
    "text_columns": TEXT_COLUMNS
}

config_path = os.path.join(ARTIFACTS_DIR, "embedding_config.json")

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"Saved configuration to: {config_path}")

In [ ]:
print("EMBEDDING PIPELINE VERDICT")

print("Documents :", len(df))
print("Dimensions:", embeddings.shape[1])
print("Model     :", EMBEDDING_MODEL)
print("Embeddings:", embedding_path)
print("Metadata  :", metadata_path)
print("Config    :", config_path)